In [1]:
import torch
from dinosaw.helpers import ModelTypes, model_names, get_models,get_features, add_custom_font
from dinosaw.utils import do_2D_pca

import numpy as np
from PIL import Image

import matplotlib.pyplot as plt

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda:1'
half = False

/home/ab_aimd_anja_20884/anaconda3/envs/test-multi-gpu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## good images

In [2]:
img_fname = "cells.jpg" # very good example
img_fname = "black_cat_drawing3.jpg" # very good example #SF=2
img_fname = "orig_cat_scetch.jpg"  # also very good SF=0.17
img_fname = "sctech_deer.jpg"  # also very good SF=0.5 or 600 by 400

## paper plot

In [3]:
enabled_models: tuple[ModelTypes, ...] = ('nope', 'alibi_dv2_coco') #'dv2',
models = get_models(enabled_models, "../../trained_models", DEVICE, half)

imgs = ("cells.jpg", "black_cat_drawing3.jpg", "sctech_deer.jpg") # "orig_cat_scetch.jpg",

In [4]:
channel_group = 0
features, features_reduced = [], {model_key: {} for model_key in enabled_models}
for img_file in imgs:
    for model_key in enabled_models:
        model = models[model_key]
        img_path = f'../images/{img_file}'
        img = Image.open(img_path).convert('RGB')
        sf_calc = 518 / img.height
        if img_file == "cells.jpg":
            sf_calc = 1
        img = img.resize((int(sf_calc * img.width), int(sf_calc * img.height)), Image.LANCZOS)
        feats = get_features(model, img, device=DEVICE, channel_last=False, to_half=half)#.cpu()
        # features.append(feats)
        features_reduced[model_key][img_file] =  do_2D_pca(feats, (channel_group+1)*3, pre_norm="std", post_norm='minmax')[:, :, channel_group*3:channel_group*3+3]

TODO: 
maybe reorder

In [5]:
%%capture
from matplotlib.gridspec import GridSpec
FS = 30
NCOLS=len(imgs)
NROWS = 1 + len(enabled_models)
H,W = 14,19

# fig, axs = plt.subplots(nrows=NROWS, ncols=NCOLS, figsize=(25, 25), gridspec_kw={"hspace":0.5})
fig = plt.figure(figsize=(W,H))
gs = GridSpec(NROWS, NCOLS, figure=fig)



for i, img_file in enumerate(imgs):
    ax = fig.add_subplot(gs[0, i])
    ax.imshow(Image.open(f"../images/{img_file}").convert("RGB"))
    ax.set_axis_off()
    for j, model_key in enumerate(enabled_models):
        ax = fig.add_subplot(gs[j+1, i])
        feats=features_reduced[model_key][img_file]
        ax.imshow(feats)
        ax.set_yticks([])
        ax.set_xticks([])
        if i==0:
            ax.set_ylabel(model_names[model_key], fontsize=FS, fontweight="bold" if "alibi" in model_key else None)
fig.tight_layout()
plt.savefig("saved/S07_NoPE_v_ALiBi.jpeg", dpi=300, bbox_inches='tight', pil_kwargs={'optimize': True})